In [ ]:
!pip install dspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 25.9 MB/s eta 0:00:00


In [ ]:
import os
import dspy
from pathlib import Path
from typing import List, Dict, Optional
from google.colab import drive, ai


# Directory paths
XML_FOLDER = "/content/drive/MyDrive/AI6129/xml"
FORMAT_RULES_FILE = "/content/drive/MyDrive/AI6129/accession_formats.md"


In [ ]:
def setup_colab_environment():
    """
    Setup Google Colab environment.
    - Mounts Google Drive
    - Configures DSPy with Colab's built-in AI (NO API KEY NEEDED)
    """
    print("="*80)
    print("SETTING UP COLAB ENVIRONMENT")
    print("="*80)

    # Mount Google Drive
    print("\n1. Mounting Google Drive...")
    try:
        drive.mount('/content/drive', force_remount=False)
        print("   Google Drive mounted")
    except Exception as e:
        print(f"  Error mounting Drive: {e}")
        return False

    # Check Colab AI availability
    print("\n2. Checking Colab AI availability...")
    try:
        # Test the built-in AI
        test_response = ai.generate_text(
            "Say 'ready'",
            model_name='google/gemini-2.0-flash-lite'
        )
        print("   Colab AI is available (Gemini models)")
        print(f"  Test response: {test_response[:50]}...")
    except Exception as e:
        print(f"  Error accessing Colab AI: {e}")
        print("   Note: Colab AI requires Pro or Pro+ subscription")
        return False

    # Configure DSPy with custom Colab AI wrapper
    print("\n3. Configuring DSPy with Colab AI...")
    try:
        lm = ColabAIWrapper()
        dspy.configure(lm=lm)
        print("  DSPy configured with Colab's built-in Gemini")
    except Exception as e:
        print(f" Error configuring DSPy: {e}")
        return False

    print("\n" + "="*80)
    print("SETUP COMPLETE")
    print("="*80 + "\n")
    return True



# ==================================================
# COLAB AI WRAPPER FOR DSPy
# ==================================================

In [ ]:
class ColabAIWrapper(dspy.LM):
    """
    Wrapper to use Google Colab's built-in AI with DSPy.
    No API key required - uses Colab Pro/Pro+ built-in models.
    """

    def __init__(self, model_name='google/gemini-2.0-flash-lite', **kwargs):
        """
        Initialize Colab AI wrapper.

        Args:
            model_name: Colab AI model to use
                - 'google/gemini-2.0-flash-lite' (default, fast)
                - 'google/gemini-2.0-flash' (more capable)
        """
        self.model_name = model_name
        self.kwargs = kwargs

        # Initialize base LM
        super().__init__(model=model_name)

    def __call__(self, prompt=None, messages=None, **kwargs):
        """
        Make a completion request using Colab AI.

        Args:
            prompt: String prompt (for simple queries)
            messages: List of message dicts (for chat format)
            **kwargs: Additional generation parameters

        Returns:
            List of strings (completions)
        """
        # Combine instance kwargs with call kwargs
        generation_kwargs = {**self.kwargs, **kwargs}

        # Convert messages to prompt if needed
        if messages:
            prompt = self._messages_to_prompt(messages)
        elif not prompt:
            raise ValueError("Either 'prompt' or 'messages' must be provided")

        # Generate using Colab AI
        try:
            response = ai.generate_text(
                prompt,
                model_name=self.model_name
            )

            # Return in format expected by DSPy
            return [response]

        except Exception as e:
            print(f"Error in Colab AI generation: {e}")
            raise

    def _messages_to_prompt(self, messages):
        """
        Convert DSPy message format to a single prompt string.

        Args:
            messages: List of message dicts with 'role' and 'content'

        Returns:
            Formatted prompt string
        """
        prompt_parts = []

        for msg in messages:
            role = msg.get('role', 'user')
            content = msg.get('content', '')

            if role == 'system':
                prompt_parts.append(f"System: {content}")
            elif role == 'user':
                prompt_parts.append(f"User: {content}")
            elif role == 'assistant':
                prompt_parts.append(f"Assistant: {content}")

        return "\n\n".join(prompt_parts)

# ===========================================================
# RESOURCE LOADING
# ===========================================================

In [ ]:
def load_format_rules(filepath: str = FORMAT_RULES_FILE) -> Optional[str]:
    """
    Load accession number format rules from markdown file.

    Args:
        filepath: Path to markdown file containing format rules

    Returns:
        String content of format rules, or None if error
    """

    print(f"Loading format rules from: {filepath}")

    if not os.path.exists(filepath):
        print(f" Format rules file not found: {filepath}")
        return None

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"Format rules loaded ({len(content)} characters)")
        return content
    except Exception as e:
        print(f"Error reading format file: {e}")
        return None

def load_single_xml(filepath: str) -> Optional[str]:
  """
    Load a single XML file.

    Args:
        filepath: Path to XML file

    Returns:
        XML content as string, or None if error
  """
  if not os.path.exists(filepath):
      print(f" XML file not found: {filepath}")
      return None

  try:
      with open(filepath, 'r', encoding='utf-8') as f:
          content = f.read()

      size_kb = len(content) / 1024
      print(f" Loaded XML ({size_kb:.1f} KB)")
      return content

  except Exception as e:
      print(f" Error reading XML file: {e}")
      return None

def load_xml_folder(folder_path: str = XML_FOLDER, max_files: Optional[int] = None) -> Dict[str, str]:
  """
  Load all XML files from a folder.

  Args:
      folder_path: Path to folder containing XML files
      max_files: Optional limit on number of files to load

  Returns:
      Dictionary mapping filename to XML content
  """

  xml_files = {}

  if not os.path.exists(folder_path):
      print(f" Folder not found: {folder_path}")
      return xml_files

  xml_file_list = list(Path(folder_path).glob("*.xml"))
  if not xml_file_list:
    print(f" No XML files found in: {folder_path}")
    return xml_files

  print(f"Found {len(xml_file_list)} XML files in folder")

  if max_files:
    xml_file_list = xml_file_list[:max_files]
    print(f" Loading only the first {max_files} files")

  for xml_file in xml_file_list:
    try:
      with open(xml_file, 'r', encoding='utf-8') as f:
        content = f.read()

      xml_files[xml_file.name] = content
      size_kb = len(content) / 1024
      print(f" {xml_file.name} ({size_kb:.1f} KB)")

    except Exception as e:
      print(f" Error reading XML file: {e}")

  print(f"Successfully loaded {len(xml_files)} files")
  return xml_files


# ==========================================================
# DSPy SIGNATURE & MODULE
# ==========================================================

In [ ]:

class AccessionExtractionSignature(dspy.Signature):
    """
    Extract GenBank accession numbers from scientific article XML.

    Your task is to find all GenBank accession numbers mentioned in the article.

    GenBank accession numbers follow specific formats defined in the format_rules.
    Look for these accession numbers in:
    - Methods sections (sample/strain information)
    - Results sections (sequence data references)
    - Tables (strain collections, sequence identifiers)
    - Supplementary data references

    For each accession found, provide:
    1. The accession number itself
    2. Brief context (what section/sentence it appears in)
    3. What it refers to (e.g., "Hepatitis A strain", "viral genome sequence")

    CRITICAL EXTRACTION RULES:
    - Extract each accession ONLY ONCE (return unique accessions only, no duplicates)
    - Do NOT extract example accessions from the articles (e.g., AB123456, XM_123456)
    - Only extract accessions that actually appear in the xml_content
    - The format_rules contain examples for reference - these are NOT real accessions
    - Do NOT extract journal article identifiers (e.g., s41588-021-00978-w)
    - Do NOT extract DOI components or reference numbers
    - GenBank accessions never contain hyphens (except RefSeq underscores)
    """

    format_rules: str = dspy.InputField(
        desc="Rules defining valid GenBank accession number formats. "
             "Contains format examples that should NOT be extracted as real accessions."
    )
    xml_content: str = dspy.InputField(
        desc="Full XML content of the scientific article"
    )
    accessions: str = dspy.OutputField(
        desc="List of UNIQUE accessions found in the article (each accession listed only once). "
             "Format: 'ACCESSION | Context | Description'. "
             "Do NOT include format examples from the rules. "
             "Only include accessions that appear in the xml_content."
    )

class AccessionExtractor(dspy.Module):
    """DSPy module for extracting accession numbers."""

    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(AccessionExtractionSignature)

    def forward(self, format_rules: str, xml_content: str):
        """
        Extract accessions using Chain of Thought reasoning.

        Args:
            format_rules: GenBank format rules
            xml_content: Article XML content

        Returns:
            DSPy prediction with accessions field
        """
        return self.extract(format_rules=format_rules, xml_content=xml_content)


# =========================================================
# EXTRACTION FUNCTIONS
# =========================================================

In [ ]:
def extract_from_single_file(xml_filepath: str, format_rules_path: str = FORMAT_RULES_FILE) -> Dict:
    """
    Extract accession numbers from a single XML file.

    Args:
        xml_filepath: Path to XML file
        format_rules_path: Path to format rules markdown

    Returns:
        Dictionary with extraction results
    """

    print("\n" + "="*80)
    print("EXTRACTING FROM SINGLE FILE")
    print("="*80 + "\n")

    #load format rules
    format_rules = load_format_rules(format_rules_path)
    if not format_rules:
      return {"error": "Failed to load format rules"}

    #Load XML file
    print(f"\nLoading XML: {xml_filepath}")
    xml_content = load_single_xml(xml_filepath)
    if not xml_content:
      return {"error": "Failed to load XML file"}

    #Extract accessions
    print("\nExtracting accession numbers using Colab AI...")
    extractor = AccessionExtractor()

    try:

        result = extractor(format_rules=format_rules, xml_content=xml_content)

        print("\n" + "-"*80)
        print("EXTRACTION RESULTS")
        print("-"*80)
        print(result.accessions)
        print("-"*80 + "\n")

        return {
            "filename": os.path.basename(xml_filepath),
            "accessions": result.accessions,
            "raw_result": result
        }

    except Exception as e:
        print(f" Extraction error: {e}")
        return {"error": str(e)}

def extract_from_folder(folder_path: str = XML_FOLDER, format_rules_path: str = FORMAT_RULES_FILE, max_files: Optional[int] = None) -> List[Dict]:
    """
    Extract accession numbers from all XML files in a folder.

    Args:
        folder_path: Path to folder containing XML files
        format_rules_path: Path to format rules markdown
        max_files: Optional limit on number of files to process

    Returns:
        List of dictionaries with extraction results per file
    """

    print("\n" + "="*80)
    print("EXTRACTING FROM FOLDER")
    print("="*80 + "\n")

    #Load format rules
    format_rules = load_format_rules(format_rules_path)
    if not format_rules:
      return {"error": "Failed to load format rules"}

    #Load XML files
    print(f"\nLoading XML files from: {folder_path}")
    xml_files = load_xml_folder(folder_path, max_files)

    if not xml_files:
      return [{"error": "No XML files loaded"}]

    #Extract from each file
    results = []
    extractor = AccessionExtractor()

    for idx, (filename, xml_content) in enumerate(xml_files.items(), 1):
        print(f"\n{'='*80}")
        print(f"Processing [{idx}/{len(xml_files)}]: {filename}")
        print(f"{'='*80}")

        try:
          result = extractor(format_rules=format_rules, xml_content=xml_content)

          print("\n" + "-"*40)
          print("RESULTS:")
          print("-"*40)
          print(result.accessions)
          print("-"*40 + "\n")

          results.append({
              "filename": filename,
              accessions: result.accessions,
              "result": result
          })
        except exception as e:
            print(f" Error processing {filename}: {e}")
            results.append({
                "filename": filename,
                "error": str(e)
            })

    print("\n" + "="*80)
    print(f"EXTRACTION COMPLETE - Processed {len(results)} files")
    print("="*80 + "\n")

    return results

Extract One specific file

In [ ]:
def example_single_file():
    """
    Example: Extract from one specific XML file.
    """
    print("\n" + "#"*80)
    print("EXAMPLE: SINGLE FILE EXTRACTION")
    print("#"*80 + "\n")

    # Specify your file path
    xml_file = "/content/drive/MyDrive/AI6129/xml/PMC11696716_20251014.xml"

    result = extract_from_single_file(xml_file)
    return result

In [ ]:
#PMC11696716_20251014.xml
if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: Ready
...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (16312 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC11696716_20251014.xml
 Loaded XML (257.7 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION R

In [ ]:
#PMC11703991_20251014.xml

if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: Ready
...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (16312 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC11703991_20251014.xml
 Loaded XML (207.1 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION R

Extract first five file

In [ ]:
def example_folder_extraction():
    """
    Example: Extract from folder with file limit.
    """
    print("\n" + "#"*80)
    print("EXAMPLE: FOLDER EXTRACTION (LIMITED)")
    print("#"*80 + "\n")

    # Extract from first 5 files in folder
    results = extract_from_folder(
        folder_path=XML_FOLDER,
        max_files=5
    )

    return results